In [1]:
import pandas as pd
import re

#Raw RPK matrix: peptides x sample IDs

cohort_df = pd.read_csv("./preprocessing/earliest_tp_rpk_avg_value_for_cohort_techreps_071626.csv").set_index("peptide")

#sample metadata with time info
meta = pd.read_csv("./preprocessing/master_sample_metadata_060626.csv")

#import df_hbdb
hbdb_df = pd.read_csv("./preprocessing/hbdb_pass_100k_techreps_averaged_071626.csv").set_index("peptide")

#load peptide protein metadata
protein_metadata = pd.read_csv("lassa_library_sequences_species_standardized_070726.csv")

In [2]:
import numpy as np

mat = cohort_df.apply(pd.to_numeric, errors="coerce").fillna(0.0) #converts every column to a numeric type, if cannot be converted, fills with Nan, but then later NaN gets converted to 0.0

#log transforming before z-scoring
log_mat = np.log10(mat + 1.0)

log_hbdb = np.log10(hbdb_df + 1.0)
log_hbdb_mean = log_hbdb.mean(axis=1)

log_hbdb_std = log_hbdb.std(axis=1, ddof=1)
#to prevent inflated z-scores set threshold above a small percentile of the nonzero std
#quantile 0.01 picks a value where roughly the bottom 1% of peptides sit at or below it. so it's
#grounded by a small group of low-variance peptides rather than a single outlier. 
floor = log_hbdb_std[log_hbdb_std > 0].quantile(0.01)
log_hbdb_std = log_hbdb_std.clip(lower=floor)

#z-score using healthy controls as reference
z_mat = log_mat.sub(log_hbdb_mean, axis=0).div(log_hbdb_std, axis=0)
z_mat = z_mat.fillna(0.0)

z_mat.to_csv('z_score_df_SL_over_US_083126.csv')

In [3]:
# -- peptide universe
all_peps = (
    z_mat.index
    .intersection(log_hbdb.index)
)

In [4]:
# -- HBDB LOO z-scores
hbdb_cols_z = list(log_hbdb.columns)
hbdb_kept = log_hbdb.loc[all_peps, hbdb_cols_z].astype(float).values  # (n_kept, n_hbdb)
n_hbdb_s = hbdb_kept.shape[1]

z_hbdb_loo = np.full_like(hbdb_kept, np.nan, dtype=float)
for i in range(n_hbdb_s):
    idx = np.delete(np.arange(n_hbdb_s), i)
    mu = hbdb_kept[:, idx].mean(axis=1)
    sd = hbdb_kept[:, idx].std(axis=1, ddof=1)
    #use the same floor as above. not recalculating floor - it's simply a stabilization constant.
    sd = np.where(sd == 0, floor, sd)
    with np.errstate(invalid='ignore', divide='ignore'):
        z_hbdb_loo[:, i] = (hbdb_kept[:, i] - mu) / sd

z_hbdb_loo_df = pd.DataFrame(z_hbdb_loo, index=all_peps, columns=hbdb_cols_z)
print(f"HBDB LOO z-scores computed: {n_hbdb_s} samples x {len(all_peps):,} peptides")

z_hbdb_loo_df.to_csv("z_hbdb_loo_results_083126.csv", index=True, header=True)

HBDB LOO z-scores computed: 87 samples x 90,132 peptides


### QC plots